# SurveilAMR — Exploratory Data Analysis

**2026 Vivli AMR Surveillance Data Challenge**  
Repository: https://github.com/Nana-Safo-Duker/Vivli-AMR-2026-Data-Challenge-SurveilAMR

Place approved raw ATLAS data at `data/raw/atlas_vivli_2004_2024.csv` after Vivli access.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'raw' / 'atlas_vivli_2004_2024.csv'
if not DATA.exists():
    DATA = ROOT / 'atlas_vivli_2004_2024.csv'
PROC = ROOT / 'data' / 'processed'

sns.set(style='whitegrid')
df = pd.read_csv(DATA, nrows=100_000, low_memory=False)
print(f'Loaded {len(df):,} isolates (sample)')
print(f'Columns: {len(df.columns)} | Years: {df["Year"].min()}–{df["Year"].max()}')
df.head()

In [ ]:
plt.figure(figsize=(10, 5))
df['Species'].value_counts().head(10).plot(kind='barh', color='steelblue')
plt.title('Top 10 Pathogens (Sample)')
plt.xlabel('Isolates')
plt.tight_layout()
plt.show()

In [ ]:
kp = df[df['Species'] == 'Klebsiella pneumoniae']
trend = kp.groupby('Year')['Meropenem_I'].apply(lambda x: (x == 'Resistant').mean() * 100)
trend.plot(marker='o', figsize=(10, 4), color='crimson')
plt.title('K. pneumoniae Meropenem Resistance (%) — Sample Window')
plt.ylabel('Resistance %')
plt.tight_layout()
plt.show()

In [ ]:
# Full-cohort processed summaries (run scripts/run_analysis.py first)
summary = json.loads((PROC / 'dataset_summary.json').read_text(encoding='utf-8'))
print('Total isolates:', summary['total_rows'])
print('Countries:', summary['n_countries'])
print('African isolates:', summary['africa_rows'])
print('K. pneumoniae Africa meropenem %:', summary['resistance_trends']['Klebsiella pneumoniae'].get('africa_resistance_pct'))

yearly = pd.read_csv(PROC / 'surveilamr_resistance_by_year.csv')
yearly[(yearly['Species']=='Klebsiella pneumoniae') & (yearly['Antibiotic']=='Meropenem')].tail(10)

In [ ]:
# Supplementary dataset summaries (run scripts/analyze_supplementary.py)
supp = json.loads((PROC / 'supplementary_datasets_summary.json').read_text(encoding='utf-8'))
print('DREAM n:', supp['dream']['n'], '| BDQ median:', supp['dream']['bdq_mic_median'])
print('KEYSTONE n:', supp['keystone']['n'], '| species:', supp['keystone']['n_species'])
print('GASAR n:', supp['gasar']['n'], '| poly MIC>=4 %:', round(supp['gasar']['poly_mic_ge4_pct'], 2))